# 1.问题分解策略

知识点讲解：问题分解策略（Decomposition / Least-to-Most Prompting）

1. 复杂问题为什么检索不准
   - 复杂问题往往包含多个子意图，例如"LLMOps 的应用配置怎么做、有哪些限制、出错怎么排查"，这是三个独立的知识点
   - 把它们压缩成一个向量后，得到的是多个语义的"平均值"，这个平均向量可能不接近任何一个具体文档，导致三个方面都召回不佳
   - 这是向量检索的固有缺陷：一个向量只能表达一个聚焦的语义

2. 问题分解策略的核心思想
   - 先让 LLM 把复杂问题拆解为若干个可以独立回答的子问题
   - 每个子问题语义聚焦，单独检索时向量匹配更精准
   - 分别回答子问题，最后把子答案汇总形成对原问题的完整回答
   - 这就是论文中的 Least-to-Most Prompting：从最简单的子问题逐步推进到复杂问题

3. 两种子问题处理模式
   - 并行模式（Individual Answer）：各子问题独立检索、独立回答，最后统一汇总。
     优点是可并发、延迟低；缺点是子答案之间缺乏关联
   - 串行模式（Recursive Answer，本例采用）：按顺序回答，
     把"已回答的 QA 对"作为上下文传给下一个子问题，形成递进推理链。
     优点是后面的回答能利用前面的结论，逻辑更连贯；缺点是无法并发、延迟叠加

4. 本例迭代问答链的三路输入（关键设计）
   - question：当前正在处理的子问题
   - qa_pairs：之前所有子问题的"问题+答案"累积文本（递进推理的载体）
   - context：针对当前子问题实时检索到的文档（RAG 的知识来源）
   - 三者共同构成 Prompt，让 LLM 既看到历史推理又看到新鲜资料

5. itemgetter 在 LCEL 中的作用
   - operator.itemgetter("question") 等价于 lambda x: x["question"]
   - 在 LCEL 并行字典中用于从上游输入 dict 中"抽取指定字段"
   - itemgetter("question") | retriever 是一个子链：
     先抽出 question 字符串，再把它作为输入喂给 retriever，
     这样就把 dict 输入适配成了 retriever 需要的 str 输入
   - 用它比 lambda 更简洁，且 LangChain 能更好地序列化和追踪

6. 分解链的输出后处理
   - LLM 返回的是多行文本，需要 (lambda x: x.strip().split("
")) 转为列表
   - 这个 lambda 会被 LCEL 自动包装为 RunnableLambda
   - 风险点：LLM 可能输出带序号的行（"1. xxx"）或包含空行，
     生产环境应做更健壮的清洗（去序号、过滤空行、限制数量）

7. 最佳实践建议
   - 分解链的 LLM 必须 temperature=0，否则子问题每次都不同，结果无法复现
   - Prompt 中明确限定子问题数量（本例为 3 个），否则 LLM 可能生成过多导致延迟爆炸
   - LLM 输出解析要健壮：过滤空行、剥离 "1." 之类的序号前缀、去重、限制上限
   - 串行模式的延迟是"子问题数 × (1 次检索 + 1 次 LLM)"，3 个子问题约 5~10 秒；
     对延迟敏感的场景改用并行模式，配合 RunnableParallel 或 abatch 并发执行
   - qa_pairs 会随迭代不断变长，需注意上下文窗口上限，必要时做摘要压缩
   - 本例只打印了各子问题的答案，生产环境还应再加一次"汇总链"，
     把所有 QA 对综合成对原始问题的最终回答
   - 简单问题不要用分解策略，会白白增加数倍成本与延迟；
     可先用一个轻量分类器判断问题复杂度，再决定是否走分解流程


In [ ]:
from operator import itemgetter
import dotenv
import weaviate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_weaviate import WeaviateVectorStore
from weaviate.auth import AuthApiKey
dotenv.load_dotenv()

In [ ]:
def format_qa_pair(question: str, answer: str) -> str:
    """格式化传递的问题+答案为单个字符串
    作用：把一组"子问题 + 子答案"拼装成结构化文本，
          作为后续子问题的历史推理上下文使用。
    参数：
        question: str - 子问题文本
        answer: str   - 该子问题的回答
    返回：
        str - 形如 "Question: xxx\nAnswer: yyy" 的字符串
              （末尾的换行被 strip() 清除，避免累积过多空行）
    设计意图：使用固定的 "Question:/Answer:" 前缀，
              让 LLM 能清晰识别这是历史问答对而非当前待答问题
    """
    return f"Question: {question}\nAnswer: {answer}\n\n".strip()

In [ ]:
decomposition_prompt = ChatPromptTemplate.from_template(
    "你是一个乐于助人的AI助理，可以针对一个输入问题生成多个相关的子问题。\n"
    "目标是将输入问题分解成一组可以独立回答的子问题或者子任务。\n"
    "生成与一下问题相关的多个搜索查询：{question}\n"
    "并使用换行符进行分割，输出（3个子问题/子查询）："
)
decomposition_chain = (
        {"question": RunnablePassthrough()}
        | decomposition_prompt
        | ChatOpenAI(model="deepseek-v4-pro", temperature=0)
        | StrOutputParser()
        | (lambda x: x.strip().split("\n"))
)
db = WeaviateVectorStore(
    client=weaviate.connect_to_wcs(
        cluster_url="https://mbakeruerziae6psyex7ng.c0.us-west3.gcp.weaviate.cloud",
        auth_credentials=AuthApiKey("ZltPVa9ZSOxUcfafelsggGyyH6tnTYQYJvBx"),
    ),
    index_name="DatasetDemo",
    text_key="text",
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"),
)
retriever = db.as_retriever(search_type="mmr")
question = "关于LLMOps应用配置的文档有哪些"
sub_questions = decomposition_chain.invoke(question)
prompt = ChatPromptTemplate.from_template("""这是你需要回答的问题：
---
{question}
---
这是所有可用的背景问题和答案对：
---
{qa_pairs}
---
这是与问题相关的额外背景信息：
---
{context}
---""")
chain = (
        {
            "question": itemgetter("question"),
            "qa_pairs": itemgetter("qa_pairs"),
            "context": itemgetter("question") | retriever,
        }
        | prompt
        | ChatOpenAI(model="deepseek-v4-pro", temperature=0)
        | StrOutputParser()
)
qa_pairs = ""
for sub_question in sub_questions:
    answer = chain.invoke({"question": sub_question, "qa_pairs": qa_pairs})
    qa_pair = format_qa_pair(sub_question, answer)
    qa_pairs += "\n---\n" + qa_pair
    print(f"问题: {sub_question}")
    print(f"答案: {answer}")